In [70]:
import pandas as pd
import sqlite3
from datetime import datetime

from zoneinfo import ZoneInfo

# Test case: Payments reconciliation — June 2026

## Context

Lumo Wellness sells app subscriptions through five payment providers: PayPal (separate US and EU accounts), dLocal (Latin America), Adyen, and Google Play.

In [71]:
# PayPal US account activity export
df_paypal_us_activity_20260601_20260703 = pd.read_csv("raw/paypal_us_activity_20260601_20260703.csv") 
df_paypal_us_activity_20260601_20260703.columns = df_paypal_us_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_paypal_us_activity_20260601_20260703["datetime_utc"] = df_paypal_us_activity_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['date']} {row['time']}").tz_localize(ZoneInfo(row["time_zone"])).tz_convert("UTC")
    , axis=1)
df_paypal_us_activity_20260601_20260703.head()


,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/01/2026,07:12:11,GMT,Taylor Brooks,7FTLSR4XAT5TKVLKF,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500884,2026-06-01 07:12:11+00:00
1,06/01/2026,07:54:32,GMT,Morgan Hayes,7R82A7SLSXJAD72J1,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500132,2026-06-01 07:54:32+00:00
2,06/01/2026,07:59:18,GMT,Drew Bennett,S1VNZT7ACC5E6HNYD,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500591,2026-06-01 07:59:18+00:00
3,06/01/2026,08:03:29,GMT,Riley Carter,BTDR7VQTQ8G4GTMGS,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500479,2026-06-01 08:03:29+00:00
4,06/01/2026,08:16:21,GMT,Casey Nguyen,NFQZ142PMP4P1BC2Q,NaN,Website Payment,Completed,USD,24.99,-1.36,23.63,ORD-500005,2026-06-01 08:16:21+00:00


In [72]:
# PayPal EU account activity export
# semicolon, cp1252 (Windows-1252)
df_paypal_eu_activity_20260601_20260703 = pd.read_csv("raw/paypal_eu_activity_20260601_20260703.csv", sep=";", encoding="cp1252") 
df_paypal_eu_activity_20260601_20260703.columns = df_paypal_eu_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_paypal_eu_activity_20260601_20260703["datetime_utc"] = df_paypal_eu_activity_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['date']} {row['time']}").tz_localize(ZoneInfo(row["time_zone"])).tz_convert("UTC")
    , axis=1)
df_paypal_eu_activity_20260601_20260703.head()

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/03/2026,12:20:00,GMT,Jürgen Müller,N2D9QMQNH8VUEES5X,NaN,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507782,2026-06-03 12:20:00+00:00
1,01/06/2026,08:17:07,GMT,Céline Marchand,ZEW9RTFHKTBFQT5E3,NaN,Website Payment,Completed,GBP,"8,99","-0,56","8,43",ORD-502669,2026-01-06 08:17:07+00:00
2,01/06/2026,08:35:02,GMT,José García,0DRQ0928NVSA52MA3,NaN,Website Payment,Denied,GBP,"4,49","0,00","0,00",ORD-503463,2026-01-06 08:35:02+00:00
3,01/06/2026,08:48:13,GMT,Nuño Ibáñez,P54742W2ULA8YDR0L,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502556,2026-01-06 08:48:13+00:00
4,01/06/2026,09:07:25,GMT,Søren Kjær,JR1SUR8G07W17ZPNR,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502537,2026-01-06 09:07:25+00:00


In [73]:
# Adyen payment accounting report
df_adyen_payment_accounting_20260601_20260703 = pd.read_csv("raw/adyen_payment_accounting_20260601_20260703.csv") 
df_adyen_payment_accounting_20260601_20260703.columns = df_adyen_payment_accounting_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_adyen_payment_accounting_20260601_20260703["datetime_utc"] = df_adyen_payment_accounting_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['creation_date']}").tz_localize(ZoneInfo(row["timezone"])).tz_convert("UTC")
    , axis=1)
df_adyen_payment_accounting_20260601_20260703.head()

,company_account,merchant_account,psp_reference,merchant_reference,record_type,creation_date,timezone,gross_currency,gross_debit,gross_credit,commission,markup,net_currency,net_debit,net_credit,batch_number,datetime_utc
0,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Authorised,2026-06-01 09:13:28,Europe/Amsterdam,USD,NaN,24.99,NaN,NaN,USD,NaN,NaN,23,2026-06-01 07:13:28+00:00
1,LumoGroup,LumoECOM_EU,8620000000000500,ORD-504450,Authorised,2026-06-01 09:15:43,Europe/Amsterdam,EUR,NaN,9.99,NaN,NaN,EUR,NaN,NaN,23,2026-06-01 07:15:43+00:00
2,LumoGroup,LumoECOM_EU,8620000000000115,ORD-504065,Authorised,2026-06-01 09:18:57,Europe/Amsterdam,EUR,NaN,24.99,NaN,NaN,EUR,NaN,NaN,23,2026-06-01 07:18:57+00:00
3,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Settled,2026-06-01 09:36:28,Europe/Amsterdam,USD,NaN,24.99,0.47,0.15,USD,NaN,24.37,23,2026-06-01 07:36:28+00:00
4,LumoGroup,LumoAPP_US,8620000000000575,ORD-504525,Authorised,2026-06-01 09:41:36,Europe/Amsterdam,USD,NaN,9.99,NaN,NaN,USD,NaN,NaN,23,2026-06-01 07:41:36+00:00


In [8]:
# dLocal transactions export
df_dlocal_transactions_20260601_20260703 = pd.read_csv("raw/dlocal_transactions_20260601_20260703.csv") 
df_dlocal_transactions_20260601_20260703.head()

,transaction_id,invoice_id,transaction_type,status,created_at_utc,country,currency,currency_exponent,local_amount,fx_rate,usd_amount,fee_usd
0,DL-90001309,ORD-507781,PAYMENT,PAID,2026-06-01 00:05:00,MX,MXN,2,12900,18.1548,7.11,0.32
1,DL-90001242,ORD-506937,PAYMENT,REJECTED,2026-06-01 07:25:52,CO,COP,2,19990000,3984.0637,50.17,2.26
2,DL-90000404,ORD-506099,PAYMENT,PAID,2026-06-01 08:17:24,MX,MXN,2,12900,18.1772,7.10,0.32
3,DL-90000750,ORD-506445,PAYMENT,PAID,2026-06-01 08:27:11,CO,COP,2,2990000,3984.0637,7.50,0.34
4,DL-90001024,ORD-506719,PAYMENT,PAID,2026-06-01 08:29:13,BR,BRL,2,3490,5.4195,6.44,0.29


In [74]:
# Google Play earnings report, June
# line 1 is a report header, data starts on line 2
df_google_play_earnings_202606 = pd.read_csv("raw/google_play_earnings_202606.csv", skiprows=1, header=0)
df_google_play_earnings_202606.columns = df_google_play_earnings_202606.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

df_google_play_earnings_202606["datetime_utc"] = df_google_play_earnings_202606.apply(
    lambda row: pd.Timestamp(f"{row['transaction_date']} {row['transaction_time']}").tz_localize(ZoneInfo("America/Los_Angeles")).tz_convert("UTC")
    , axis=1)
df_google_play_earnings_202606.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency,datetime_utc
0,"Jun 1, 2026",00:28:46,Charge,sub_annual,US,USD,59.99,1.0,USD,59.99,2026-06-01 07:28:46+00:00
1,"Jun 1, 2026",00:28:46,Google fee,sub_annual,US,USD,-9.00,1.0,USD,-9.00,2026-06-01 07:28:46+00:00
2,"Jun 1, 2026",00:53:08,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99,2026-06-01 07:53:08+00:00
3,"Jun 1, 2026",00:53:08,Google fee,sub_monthly,US,USD,-1.50,1.0,USD,-1.50,2026-06-01 07:53:08+00:00
4,"Jun 1, 2026",01:35:01,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99,2026-06-01 08:35:01+00:00


In [75]:
# Google Play earnings report, July 1–3
df_google_play_earnings_202607_partial = pd.read_csv("raw/google_play_earnings_202607_partial.csv", skiprows=1, header=0) 
df_google_play_earnings_202607_partial.columns = df_google_play_earnings_202607_partial.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

df_google_play_earnings_202607_partial["datetime_utc"] = df_google_play_earnings_202607_partial.apply(
    lambda row: pd.Timestamp(f"{row['transaction_date']} {row['transaction_time']}").tz_localize(ZoneInfo("America/Los_Angeles")).tz_convert("UTC")
    , axis=1)
df_google_play_earnings_202607_partial.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency,datetime_utc
0,"Jul 1, 2026",02:40:10,Charge refund,sub_monthly,DE,EUR,-9.99,1.085329,USD,-10.84,2026-07-01 09:40:10+00:00
1,"Jul 1, 2026",03:50:07,Charge,sub_monthly,DE,EUR,9.99,1.085329,USD,10.84,2026-07-01 10:50:07+00:00
2,"Jul 1, 2026",03:50:07,Google fee,sub_monthly,DE,EUR,-1.50,1.085329,USD,-1.63,2026-07-01 10:50:07+00:00
3,"Jul 1, 2026",05:43:32,Charge,sub_monthly,US,USD,9.99,1.000000,USD,9.99,2026-07-01 12:43:32+00:00
4,"Jul 1, 2026",05:43:32,Google fee,sub_monthly,US,USD,-1.50,1.000000,USD,-1.50,2026-07-01 12:43:32+00:00


payment_engine_log.csv is an extract from the payment backend — the engine that initiates and tracks every transaction attempt, applies FX conversion, and is treated internally as the source of truth. The other files are data exports from each payment provider’s admin panel — it’s what the PSPs themselves say happened.

In [3]:
# Payment backend (engine) log extract, all providers
df_payment_engine_log = pd.read_csv("raw/payment_engine_log.csv") 
df_payment_engine_log.head()

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107955,ORD-507781,SALE,settled,dlocal,DL-90001309,sub_monthly,MX,MXN,129.00,0.055082,2026-05-29,7.11,2026-05-31T23:59:50Z,2026-06-01T00:05:00Z
1,TXN-100884,ORD-500884,SALE,settled,paypal_us,7FTLSR4XAT5TKVLKF,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T07:03:11Z,2026-06-01T07:12:11Z
2,TXN-106445,ORD-506445,SALE,settled,dlocal,DL-90000750,sub_monthly,CO,COP,29900.00,0.000251,2026-06-01,7.50,2026-06-01T07:11:11Z,2026-06-01T08:27:11Z
3,TXN-106099,ORD-506099,SALE,settled,dlocal,DL-90000404,sub_monthly,MX,MXN,129.00,0.055014,2026-06-01,7.10,2026-06-01T07:11:24Z,2026-06-01T08:17:24Z
4,TXN-105905,ORD-505905,SALE,settled,dlocal,DL-90000210,sub_quarterly,BR,BRL,89.90,0.184519,2026-06-01,16.59,2026-06-01T07:12:27Z,2026-06-01T08:53:27Z


In [18]:
# Daily FX rates to USD (usd_rate = USD per 1 unit of currency)
df_fx_rates = pd.read_csv("raw/fx_rates.csv") 
df_fx_rates.head()

,rate_date,currency,usd_rate,published_at
0,2026-05-25,ARS,0.000839,2026-05-25T16:00:00Z
1,2026-05-25,BRL,0.184871,2026-05-25T16:00:00Z
2,2026-05-25,CLP,0.001060,2026-05-25T16:00:00Z
3,2026-05-25,COP,0.000248,2026-05-25T16:00:00Z
4,2026-05-25,EUR,1.093525,2026-05-25T16:00:00Z


In [19]:
# Contracted provider fees, where we have a contract on file
df_fee_schedule = pd.read_csv("raw/fee_schedule.csv") 
df_fee_schedule.head()

,psp,account,valid_from,percent_fee,fixed_fee,fixed_fee_currency
0,adyen,LumoECOM_EU,2026-01-01,2.5,0.0,NaN
1,adyen,LumoAPP_US,2026-01-01,2.5,0.0,NaN
2,google_play,LumoPlayMain,2026-01-01,15.0,0.0,NaN


Every month, Finance Analyst must prove that the payment backend and the provider exports tell the same story, and explain every difference in dollars if it’s not the case. That is your task, for the reporting period June 2026.

The engine-log extract and the provider exports intentionally cover a few days around the month boundary as well.


In [76]:
# connect to a database
conn = sqlite3.connect("db_simple_life.db")

df_payment_engine_log.to_sql('payment_engine_log', conn, if_exists="replace", index=False)

df_paypal_us_activity_20260601_20260703.to_sql('paypal_us_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_paypal_eu_activity_20260601_20260703.to_sql('paypal_eu_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_adyen_payment_accounting_20260601_20260703.to_sql('adyen_payment_accounting_20260601_20260703', conn, if_exists="replace", index=False)
df_dlocal_transactions_20260601_20260703.to_sql('dlocal_transactions_20260601_20260703', conn, if_exists="replace", index=False)
df_google_play_earnings_202606.to_sql('google_play_earnings_202606', conn, if_exists="replace", index=False)
df_google_play_earnings_202607_partial.to_sql('google_play_earnings_202607_partial', conn, if_exists="replace", index=False)
df_fx_rates.to_sql('fx_rates', conn, if_exists="replace", index=False)
df_fee_schedule.to_sql('fee_schedule', conn, if_exists="replace", index=False)


3

## Your task
- Reconcile each provider export against the payment engine log for June 2026. Quantify all discrepancies — in row counts and in USD.
- Classify every discrepancy by root cause, using your own taxonomy.
- Deliver a summary at provider × cause grain: row count, absolute USD impact, signed USD impact, and each cause’s share of the total discrepancy in %.
- Where fee_schedule.csv is applicable to  a provider, reconcile the reported fees against the contracted rates.
- Report the total discrepancy and show what remains unexplained after your classification (the residual). A correct analysis explains all the discrepancies.

## Evaluation notes
- Use DB of your choice (but tell us which), we assess the logic, not the dialect.
- When the data is ambiguous, make a call and document it. Documented assumptions count in your favor.
- Every dollar of difference between the payment backend and the providers is either matched, explained with a cause, or explicitly listed as unexplained — and the three add up.
- The reconciliation summary should read like something you could hand to a CFO: numbers first, caveats and issues stated.

## What to hand back
- Your SQL
- The summary from step 3, as a query result (CSV).
- A short written note: your assumptions, data-quality findings, and action items you would take to fix upstream.


In [78]:
# query0 = """
# SELECT *
# -- FROM payment_engine_log -- +
# -- FROM paypal_us_activity_20260601_20260703 -- +
# -- FROM paypal_eu_activity_20260601_20260703 -- +
# -- FROM adyen_payment_accounting_20260601_20260703 -- +
# -- FROM dlocal_transactions_20260601_20260703 -- +
# FROM google_play_earnings_202606 -- +
# -- FROM google_play_earnings_202607_partial -- +
# -- FROM fx_rates -- +
# -- FROM fee_schedule -- +
# LIMIT 100;
# """
# result0 = pd.read_sql(query0, conn)
# result0

In [54]:
query = """
--WITH adyen as (
--    SELECT 
--    FROM adyen_payment_accounting_20260601_20260703
--    WHERE
--)

SELECT 
    -- sku?? settled?? operation_type??
    psp, 
    country, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
GROUP BY 1,2,3
LIMIT 100;
"""

In [55]:
result = pd.read_sql(query, conn)
result

,psp,country,currency,amount_local,amount_usd
0,adyen,DE,EUR,10269.94,11174.15
1,adyen,FR,EUR,4924.89,5358.69
2,adyen,GB,GBP,4941.21,6383.63
3,adyen,NL,EUR,6646.49,7236.50
4,adyen,US,USD,6941.67,6941.67
5,dlocal,AR,ARS,1300490.50,1086.98
6,dlocal,BR,BRL,40246.37,7491.96
7,dlocal,CL,CLP,2057005.00,2184.89
8,dlocal,CO,COP,7891100.00,201797.80
9,dlocal,MX,MXN,77300.00,5019.47


- payment_engine_log - created_at_utc
- paypal (eu, us) - GMT
- adyen - Europe/Amsterdam
- dlocal - created_at_utc
- google play - All dates and times are in America/Los_Angeles

In [67]:
query_paypal = """
WITH paypal as (
    SELECT 
        time_zone,
        currency				
        , sum(gross) as gross
        , sum(fee) as fee
        , sum(net) as net
    FROM paypal_eu_activity_20260601_20260703
    -- FROM paypal_us_activity_20260601_20260703
    -- WHERE date(date) >= date('2026-06-01') AND date(date) < date('2026-07-01') -- timezone ???
    GROUP BY 1,2
)

SELECT *
FROM paypal
LIMIT 100;
"""
result_paypal = pd.read_sql(query_paypal, conn)
result_paypal

,time_zone,currency,gross,fee,net
0,GMT,EUR,29217.0,-733.0,26324.0
1,GMT,GBP,4585.0,-29.0,4074.0


In [68]:
query_adyen = """
WITH adyen as (
    SELECT 
        timezone,
        gross_currency,
        net_currency
        , sum(gross_credit) as gross_credit
        , sum(gross_debit) as gross_debit
        , sum(commission) as commission
        , sum(net_credit) as net_credit
        , sum(net_debit) as net_debit
    FROM adyen_payment_accounting_20260601_20260703
    WHERE creation_date >= '2026-06-01' AND creation_date < '2026-07-01' -- timezone ???
    GROUP BY 1,2,3
)

SELECT *
FROM adyen
LIMIT 100;
"""
result_adyen = pd.read_sql(query_adyen, conn)
result_adyen

,timezone,gross_currency,net_currency,gross_credit,gross_debit,commission,net_credit,net_debit
0,Europe/Amsterdam,EUR,EUR,40884.91,362.34,367.59,18862.21,362.34
1,Europe/Amsterdam,GBP,GBP,9506.61,30.98,87.28,4482.37,30.98
2,Europe/Amsterdam,USD,USD,13168.64,49.97,119.32,6119.94,49.97


In [62]:
query_dlocal = """
WITH dlocal as (
    SELECT 
        country,
        currency
        , sum(local_amount) as local_amount
        , sum(usd_amount) as usd_amount
        , sum(usd_amount * fx_rate) as fx_
    FROM dlocal_transactions_20260601_20260703
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01' -- timezone ???
    GROUP BY 1,2
)

SELECT *
FROM dlocal
LIMIT 100;
"""
result_dlocal = pd.read_sql(query_dlocal, conn)
result_dlocal

,country,currency,local_amount,usd_amount,fx_
0,AR,ARS,130049050,1087.00,1.300561e+06
1,BR,BRL,4028635,7426.49,4.028650e+04
2,CL,CLP,2050015,2177.47,2.050012e+06
3,CO,COP,789110000,1946.58,7.892381e+06
4,MX,MXN,7755800,4278.79,7.756637e+04
